# Unit 2：ICA 算法原理 —— FastICA 详解

## 目标
- 理解 ICA 的目标函数：非高斯性最大化
- 掌握 FastICA 的迭代过程
- 理解白化（whitening）的作用


### 2.1 为什么「非高斯性」是关键？

**中心极限定理**：多个独立随机变量之和的分布，比任何一个单变量的分布**更高斯**。

混合信号 $X = AS$ 是多个源信号的加权和 → 比任何单源信号都更「高斯」。

所以 ICA 的思路是：**找到让信号「最非高斯」的方向 → 那就是源信号的方向。**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kurtosis

np.random.seed(42)
n = 10000

# 三种分布：均匀、拉普拉斯、高斯
x_uniform = np.random.uniform(-1, 1, n)
x_laplace = np.random.laplace(0, 1, n)
x_gaussian = np.random.randn(n)

# 混合后（中心极限定理的体现）
x_mix = (x_uniform + x_laplace) / 2

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, data, title in zip(axes.flat,
    [x_uniform, x_laplace, x_gaussian, x_mix],
    ['均匀分布', '拉普拉斯分布', '高斯分布', '混合信号 (均匀+拉普拉斯)']):
    ax.hist(data, bins=80, density=True, alpha=0.7, color='steelblue')
    ax.set_title(f'{title}\n峰度(Kurtosis) = {kurtosis(data):.2f}')
plt.suptitle('不同分布的峰度对比（高斯=0，非高斯偏离0）', fontsize=14)
plt.tight_layout()
plt.show()

print("关键：混合信号峰度 → 0（更高斯），ICA 通过最大化非高斯性分离信号")


### 2.2 数据预处理之一：中心化

减去均值，让数据零中心，简化后续计算。


In [ ]:
# 沿用 Unit 1 的模拟数据
fs = 2000; t = np.arange(0, 1, 1/fs)
s1 = np.sin(2 * np.pi * 100 * t)
s2 = np.sin(2 * np.pi * 300 * t + 1.5)
s3 = np.sign(np.sin(2 * np.pi * 500 * t))
S = np.vstack([s1, s2, s3])

np.random.seed(42)
A = np.random.randn(3, 3)
X = A @ S

X_centered = X - X.mean(axis=1, keepdims=True)
print(f"中心化前均值: {np.array2string(X.mean(axis=1), precision=6)}")
print(f"中心化后均值: {np.array2string(X_centered.mean(axis=1), precision=6)}")
print("→ 均值归零 ✓")


### 2.3 数据预处理之二：白化（Whitening）

白化 = 去相关 + 方差归一化。变换后满足：
$$E[Z Z^T] = I$$

即各通道方差为 1，互协方差为 0。

**为什么要白化？** 它把 ICA 的搜索空间从一个任意矩阵降为**正交矩阵**，大幅简化问题。


In [ ]:
def whiten(X):
    """手工实现白化，帮助理解原理"""
    cov = np.cov(X)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # 白化矩阵 = D^(-1/2) @ E^T
    D_inv_sqrt = np.diag(1.0 / np.sqrt(eigenvalues + 1e-10))
    whitening_matrix = D_inv_sqrt @ eigenvectors.T
    return whitening_matrix @ X, whitening_matrix

Z, W_white = whiten(X_centered)

# 验证
cov_Z = np.cov(Z)
print("白化后协方差矩阵（应接近单位矩阵）:")
print(np.array2string(cov_Z, precision=4, suppress_small=True))

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_centered[0, :500], X_centered[1, :500], alpha=0.5, s=3)
axes[0].set_title('白化前（相关 + 方差不同）')
axes[0].set_aspect('equal')
axes[1].scatter(Z[0, :500], Z[1, :500], alpha=0.5, s=3, color='green')
axes[1].set_title('白化后（去相关 + 等方差 → 球状分布）')
axes[1].set_aspect('equal')
fig.suptitle('白化使数据球化', fontsize=14)
plt.tight_layout()
plt.show()


### 2.4 FastICA 核心：非高斯性度量

FastICA 用**负熵近似**作为非高斯性度量：

$$J(y) \propto [E\{G(y)\} - E\{G(\nu)\}]^2$$

其中 $\nu$ 是标准高斯变量，$G$ 是非二次函数。

常用 $G$ 函数：
| 函数 | $G(u)$ | 适用场景 |
|------|--------|---------|
| logcosh | $\frac{1}{a}\log\cosh(au)$ | 通用（默认） |
| exp | $-\exp(-u^2/2)$ | 超高斯（尖峰分布） |
| cube | $u^4/4$ | 基于峰度，对异常值敏感 |


In [ ]:
from sklearn.decomposition import FastICA
import time

for fun in ['logcosh', 'exp', 'cube']:
    start = time.time()
    ica = FastICA(n_components=3, fun=fun, random_state=42, max_iter=2000)
    S_hat = ica.fit_transform(X.T).T
    elapsed = time.time() - start
    print(f"G={fun:8s}  迭代: {ica.n_iter_:3d}  耗时: {elapsed:.4f}s")

print("\n默认 logcosh 兼顾速度与鲁棒性。")


### 2.5 FastICA 迭代过程（简化版）

核心循环（单个成分的提取）：
1. 随机初始化权重向量 $w$
2. $w^+ = E\{Z \cdot g(w^T Z)\} - E\{g'(w^T Z)\} \cdot w$
3. $w = w^+ / \|w^+\|$（归一化）
4. 与已找到的成分正交化（去相关）
5. 重复直到收敛


In [ ]:
def fastica_single_component(Z, max_iter=1000, tol=1e-6):
    """提取单个独立成分的简化 FastICA 实现"""
    n, m = Z.shape
    w = np.random.randn(n)
    w = w / np.linalg.norm(w)

    for i in range(max_iter):
        w_old = w.copy()
        # 核心不动点迭代
        wX = w @ Z
        g = np.tanh(wX)           # G' = tanh（logcosh 的导数）
        gp = 1 - np.tanh(wX)**2   # G'' = 1 - tanh²
        w_new = (Z @ g) / m - np.mean(gp) * w
        w_new = w_new / np.linalg.norm(w_new)
        w = w_new
        if abs(abs(w @ w_old) - 1) < tol:
            break
    return w, i + 1

w_ic1, iters = fastica_single_component(Z)
print(f"提取第一个独立成分，{iters} 次迭代收敛")
print(f"权重向量: {np.array2string(w_ic1, precision=4)}")


### 2.6 完整 ICA 流程图

```
原始数据 X
    │
    ▼
中心化 → X_centered（均值=0）
    │
    ▼
白化   → Z（协方差=I，球状分布）
    │
    ▼
FastICA 迭代
  ├─ 随机初始化 w
  ├─ 不动点迭代 w ← f(w)
  ├─ Gram-Schmidt 正交化（避免重复收敛）
  └─ 收敛判断
    │
    ▼
解混矩阵 W → 独立成分 S = W @ Z
```

### 2.7 成分数量怎么选？

在 EEG 中，成分数量 $k$ 的选择是一个权衡：

| $k$ 太少 | $k$ 太多 |
|----------|---------|
| 伪迹和脑信号混在同一成分中 | 脑信号过度拆分，出现噪声成分 |
| 剔除伪迹时误伤脑信号 | 人工识别负担增大 |

**经验法则：** 取通道数的 1/4 到 1/2。60 通道 EEG → 15-30 个成分。

### 🤔 思考题

- 不白化直接跑 ICA 会怎样？（试试跳过白化步骤）
- 为什么每次 FastICA 运行结果可能略有不同？

→ 进入 **Unit 3：EEG 伪迹识别**
